In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Math

# ==============================================================================
# PROBLEM 7: Symbolic Z-Transform & Interactive Pole-Zero / ROC Visualization
# Signal: x[n] = alpha^|n|, (0 < alpha < 1)
# ==============================================================================

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Solution Overview (Symbolic Calculation via SymPy)</b><br>
* <b>Signal:</b> x[n] = α<sup>|n|</sup><br>
* <b>Z-Transform Definition:</b> X(z) = Σ x[n] z<sup>-n</sup> from -∞ to +∞<br>
* <b>Poles & Zeros:</b> Computed symbolically from the resulting rational expression.<br>
* <b>ROC (Region of Convergence):</b> Ring-shaped region determined by convergence conditions.<br>
* <b>Note:</b> Use the slider to dynamically change parameter α.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

# Ορισμός συμβολικών μεταβλητών στο SymPy
n_sym = sp.Symbol('n', integer=True)
z_sym = sp.Symbol('z', complex=True)
alpha_sym = sp.Symbol('alpha', positive=True, real=True)

# 1. Συμβολικός υπολογισμός Z-Transform με τη συνάρτηση summation
x_n_sym = alpha_sym**sp.Abs(n_sym)
try:
    X_z_sym = sp.summation(x_n_sym * z_sym**(-n_sym), (n_sym, -sp.oo, sp.oo))
except Exception:
    # Αν το SymPy δυσκολευτεί απέραντα, ορίζουμε την κλειστή μορφή που προκύπτει θεωρητικά
    X_z_sym = (alpha_sym * z_sym) / (1 - alpha_sym * z_sym) + 1 / (1 - alpha_sym * z_sym**(-1))

display(Math(f"X(z) = {sp.latex(X_z_sym)}"))

n_samples = 31
n_vec = np.arange(-15, 16)

def plot_problem_7(alpha_val):
    with out:
        clear_output(wait=True)
        
        fig, (ax_pz, ax_time) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [1, 2]})
        plt.subplots_adjust(wspace=0.25)

        # --- 1. Pole-Zero Map & ROC ---
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-2.5, 2.5)
        ax_pz.set_ylim(-2.5, 2.5)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        inner_radius = alpha_val
        outer_radius = 1.0 / alpha_val

        # Σκίαση ROC (Δακτύλιος: alpha < |z| < 1/alpha)
        x_vals = np.linspace(-3.0, 3.0, 400)
        y_vals = np.linspace(-3.0, 3.0, 400)
        X, Y = np.meshgrid(x_vals, y_vals)
        Z_dist = np.sqrt(X**2 + Y**2)
        roc_mask = (Z_dist > inner_radius) & (Z_dist < outer_radius)

        ax_pz.imshow(roc_mask, extent=(-3.0, 3.0, -3.0, 3.0), origin='lower', cmap='Greens', alpha=0.25, zorder=0)

        # Όρια ROC
        theta = np.linspace(0, 2*np.pi, 200)
        ax_pz.plot(inner_radius * np.cos(theta), inner_radius * np.sin(theta), 'g:', linewidth=2, label=f'Inner ROC (|z| = α)')
        ax_pz.plot(outer_radius * np.cos(theta), outer_radius * np.sin(theta), 'g:', linewidth=2, label=f'Outer ROC (|z| = 1/α)')

        # Μοναδιαίος Κύκλος
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5)

        # Μηδενικά (Zeros)
        zeros_x = [alpha_val, -alpha_val]
        zeros_y = [0, 0]
        ax_pz.scatter(zeros_x, zeros_y, s=120, facecolors='none', edgecolors='b', linewidths=2, marker='o')

        # Πόλοι (Poles)
        poles_x = [alpha_val, outer_radius]
        poles_y = [0, 0]
        ax_pz.scatter(poles_x, poles_y, s=140, color='purple', marker='x', linewidths=3)

        ax_pz.set_title(f'Pole-Zero Map & ROC ({inner_radius:.2f} < |z| < {outer_radius:.2f})', fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)

        # Υπόμνημα
        unit_circle_handle = plt.Line2D([0], [0], color='k', linestyle='--', alpha=0.5, label='Unit Circle')
        pole_handle = plt.Line2D([0], [0], marker='x', color='purple', markersize=8, markeredgewidth=3, linestyle='None', label='Poles (α, 1/α)')
        zero_handle = plt.Line2D([0], [0], marker='o', markerfacecolor='none', markeredgecolor='b', markersize=8, markeredgewidth=2, linestyle='None', label='Zeros (±α)')
        ax_pz.legend(handles=[unit_circle_handle, pole_handle, zero_handle], loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8)

        # --- 2. Time Domain Plot ---
        x_n_vals = alpha_val**(np.abs(n_vec))

        ax_time.stem(n_vec, x_n_vals, linefmt='r-', markerfmt='ro', basefmt='k-')
        ax_time.set_title('Temporal Evolution: x[n] = α^|n|', fontsize=10, fontweight='bold')
        ax_time.set_xlabel('Time index n', fontsize=9)
        ax_time.set_ylabel('x[n]', fontsize=9)
        ax_time.set_xlim(-16, 16)
        
        max_abs_val = np.max(np.abs(x_n_vals))
        y_limit = max(1.2, min(max_abs_val * 1.25, 5.0))
        ax_time.set_ylim(-0.1, y_limit)
        ax_time.grid(True, linestyle=':', alpha=0.7)

        plt.show()

# Widget Slider
alpha_slider = widgets.FloatSlider(value=0.6, min=0.1, max=0.95, step=0.01, description='Alpha:', style={'description_width': 'initial'})

plot_problem_7(alpha_slider.value)

interactive_plot = widgets.interactive(plot_problem_7, alpha_val=alpha_slider)
display(widgets.VBox([interactive_plot, out]))